<a href="https://colab.research.google.com/github/wanjir0/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/duka_sales_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Date parsing**

In [1]:
import numpy as np

import pandas as pd



rng = np.random.default_rng(2026)

n = 1000

products = {"Phone": 15000, "Laptop": 65000, "Headphones": 2500,
"Tablet": 28000, "Charger": 1200}



sales = pd.DataFrame({

"order_id": range(1001, 1001 + n),
"order_date": pd.to_datetime("2026-01-01")

+ pd.to_timedelta(rng.integers(0, 180, n), unit="D"),

"branch": rng.choice(["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"], n,

p=[0.4, 0.2, 0.15, 0.15, 0.1]),
"product": rng.choice(list(products), n),

"quantity": rng.integers(1, 6, n),

"channel": rng.choice(["In-store", "Online", "Phone order"], n,

p=[0.55, 0.35, 0.10]),

})

sales["unit_price"] = sales["product"].map(products)

sales.to_csv("duka_sales.csv", index=False)

In [2]:
import pandas as pd

df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nMissing values per column:\n", df.isna().sum())
print("\nDate range:", df["order_date"].min().date(), "to", df["order_date"].max().date())
print("\nDuplicate order_ids:", df["order_id"].duplicated().sum())

for col in ["branch", "product", "channel"]:
    print(f"\n{col}:\n", df[col].value_counts())

df.head()

Shape: (1000, 7)

Dtypes:
 order_id               int64
order_date    datetime64[ns]
branch                object
product               object
quantity               int64
channel               object
unit_price             int64
dtype: object

Missing values per column:
 order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

Date range: 2026-01-01 to 2026-06-29

Duplicate order_ids: 0

branch:
 branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117
Name: count, dtype: int64

product:
 product
Laptop        220
Tablet        215
Phone         194
Headphones    187
Charger       184
Name: count, dtype: int64

channel:
 channel
In-store       579
Online         320
Phone order    101
Name: count, dtype: int64


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


**First-five-minutes inspection**

1.  **Shape:** 1,000 rows × 7 columns, one row per order (each order is a single product line).
2. **Types:** `order_date` parsed as datetime; `order_id`, `quantity`, `unit_price` are integers; `branch`, `product`, `channel` are text categories.
3. **Missing values:** none in any column. **Duplicate order IDs:** none.
4 **Date range:** 1 Jan 2026 to 29 Jun 2026, so the half-year is covered (there are no orders on 30 June).
5. **Key categories:** 5 branches (Nairobi largest with 376 orders), 5 products (Phone, Laptop, Headphones, Tablet, Charger), 3 channels (In-store 579, Online 320, Phone order 101). Quantity runs from 1 to 5.
6. **Note:** there is no `revenue` column yet, so it has to be derived (quantity × unit_price) in the next step.




**creating columns**

In [3]:
import numpy as np

df["revenue"] = df["quantity"] * df["unit_price"]
df["month"] = df["order_date"].dt.month_name()
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5    # Mon=0 ... Sat=5, Sun=6
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")

df[["order_id", "order_date", "revenue", "month", "weekday", "is_weekend", "price_band"]].head()

,order_id,order_date,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,45000,June,Wednesday,False,Value
1,1002,2026-02-02,260000,February,Monday,False,Premium
2,1003,2026-01-05,140000,January,Monday,False,Premium
3,1004,2026-04-26,12500,April,Sunday,True,Value
4,1005,2026-03-07,7500,March,Saturday,True,Value


**derived columns**

1.   `revenue` = quantity × unit_price (KES per order).
2. `month` and `weekday` are names taken from `order_date`.
3.  `is_weekend` is True for Saturday and Sunday.
4. `price_band` is Premium when the **unit price** is at least KES 28,000 (Tablet and Laptop), otherwise Value. It is based on unit price, not order revenue, so a 5-unit Charger order is still Value.



In [4]:
branch = (df.groupby("branch")
            .agg(total_revenue=("revenue", "sum"), orders=("order_id", "count"))
            .assign(avg_order_revenue=lambda d: d["total_revenue"] / d["orders"]))

branch["rank_revenue"] = branch["total_revenue"].rank(ascending=False).astype(int)
branch["rank_orders"] = branch["orders"].rank(ascending=False).astype(int)
branch = branch.sort_values("total_revenue", ascending=False)

print("Rankings identical:", (branch["rank_revenue"] == branch["rank_orders"]).all())
branch

Rankings identical: True


,total_revenue,orders,avg_order_revenue,rank_revenue,rank_orders
branch,,,,,
Nairobi,26106100,376,69431.117021,1,1
Mombasa,14003700,186,75288.709677,2,2
Nakuru,11970400,162,73891.358025,3,3
Kisumu,10868100,159,68352.830189,4,4
Eldoret,8844200,117,75591.452991,5,5


**Branch ranking: revenue vs orders**

**The rankings are the same:** Nairobi, Mombasa, Nakuru, Kisumu, Eldoret on both measures.

**Why:** revenue = orders × average order revenue. Average order revenue is similar in every branch (about KES 68k to 76k) because every branch sells the same product mix at the same prices. What really differs is order volume (Nairobi 376 orders vs Eldoret 117). The volume gaps are large and the average-value gaps are small, so volume decides the ranking.

**Caveat:** Nakuru and Kisumu are close (162 vs 159 orders). Nakuru stays ahead on revenue partly because its average order is higher (KES 73.9k vs 68.4k), so that position could flip on different data.

In [5]:
prod = (df.groupby("product")
          .agg(total_revenue=("revenue", "sum"), units=("quantity", "sum"))
          .assign(unit_price=pd.Series(products)))

print("Top 3 by revenue:\n", prod.nlargest(3, "total_revenue")[["total_revenue", "units"]])
print("\nTop 3 by units:\n", prod.nlargest(3, "units")[["units", "total_revenue"]])

prod.sort_values("total_revenue", ascending=False)

Top 3 by revenue:
          total_revenue  units
product                      
Laptop        41535000    639
Tablet        19348000    691
Phone          8880000    592

Top 3 by units:
          units  total_revenue
product                      
Tablet     691       19348000
Laptop     639       41535000
Phone      592        8880000


,total_revenue,units,unit_price
product,,,
Laptop,41535000,639,65000
Tablet,19348000,691,28000
Phone,8880000,592,15000
Headphones,1387500,555,2500
Charger,642000,535,1200


**Top three products:revenue vs units**

**by revenue:**1. Laptop (KES 41.5M), 2. Tablet (KES 19.3M), 3. Phone (KES 8.9M).

**by units sold:** 1. Tablet (691), 2. Laptop (639), 3. Phone (592).

**The same three products appear in both lists, but Laptop and Tablet swap places.** Tablet sells more units than Laptop (691 vs 639), but a Laptop costs KES 65,000 against KES 28,000, so Laptop earns about twice the revenue. Revenue depends on price × volume, while units count volume only.

Headphones (555 units) and Chargers (535) sell almost as many units as Phones (592). Their low prices (KES 2,500 and KES 1,200) keep them out of the revenue top three, and together they make up under 3% of revenue.


In [6]:
wk = (df.groupby("is_weekend")
        .agg(orders=("order_id", "count"),
             total_revenue=("revenue", "sum"),
             avg_order_revenue=("revenue", "mean"))
        .rename(index={False: "Weekday", True: "Weekend"}))

# Context: how many calendar days of each type are in the period?
days = pd.date_range(df["order_date"].min(), df["order_date"].max())
weekend_days = (days.dayofweek >= 5).sum()
wk["calendar_days"] = [len(days) - weekend_days, weekend_days]
wk["orders_per_day"] = wk["orders"] / wk["calendar_days"]

wk

,orders,total_revenue,avg_order_revenue,calendar_days,orders_per_day
is_weekend,,,,,
Weekday,723,51856200,71723.651452,128,5.648438
Weekend,277,19936300,71972.202166,52,5.326923


**compares weekend vs weekday orders**
| | Orders | Total revenue (KES) | Average order revenue (KES) |
|---|---|---|---|
| Weekday | 723 | 51.9M | 71,724 |
| Weekend | 277 | 19.9M | 71,972 |

Weekdays have more orders and more revenue, but that is mostly because there are more weekdays: 128 weekdays against 52 weekend days in the period. Per calendar day, weekends average 5.3 orders against 5.6 on weekdays, about 6% lower. Average order value is almost identical (KES 72.0k vs 71.7k). So the weekend effect is small, and customers don't spend more or less per order on weekends.

In [7]:
monthly = df.groupby(df["order_date"].dt.to_period("M"))["revenue"].sum()

best_m, worst_m = monthly.idxmax(), monthly.idxmin()
best_v, worst_v = monthly.max(), monthly.min()

print(monthly.map("{:,.0f}".format))
print(f"\nBest:  {best_m}  KES {best_v:,.0f}")
print(f"Worst: {worst_m}  KES {worst_v:,.0f}")
print(f"Best is {(best_v - worst_v) / worst_v:.1%} higher than worst")
print(f"Worst is {(best_v - worst_v) / best_v:.1%} lower than best")

order_date
2026-01    11,786,600
2026-02    11,772,000
2026-03    12,396,800
2026-04    13,518,300
2026-05    11,415,400
2026-06    10,903,400
Freq: M, Name: revenue, dtype: object

Best:  2026-04  KES 13,518,300
Worst: 2026-06  KES 10,903,400
Best is 24.0% higher than worst
Worst is 19.3% lower than best


###**Best and worst month**
- **Best month:** April 2026, KES 13,518,300.
- **Worst month:** June 2026, KES 10,903,400.
- **Difference:** April is **24.0% higher** than June. Put the other way, June is 19.3% lower than April.

The data has no orders on 30 June, so June covers 29 days of orders.

In [8]:
mask = ((df["price_band"] == "Premium")
        & (df["channel"] == "Online")
        & (df["branch"].isin(["Nairobi", "Mombasa"])))

premium_online = df.loc[mask]

print("Orders:", len(premium_online))
print(f"Total revenue: KES {premium_online['revenue'].sum():,.0f}")

premium_online.to_csv("premium_online_coast_capital.csv", index=False)
premium_online.head()

Orders: 83
Total revenue: KES 10,218,000


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
6,1007,2026-01-15,Nairobi,Laptop,2,Online,65000,130000,January,Thursday,False,Premium
13,1014,2026-06-12,Nairobi,Tablet,3,Online,28000,84000,June,Friday,False,Premium
32,1033,2026-04-29,Nairobi,Laptop,2,Online,65000,130000,April,Wednesday,False,Premium
35,1036,2026-03-02,Nairobi,Laptop,4,Online,65000,260000,March,Monday,False,Premium
39,1040,2026-02-10,Mombasa,Laptop,5,Online,65000,325000,February,Tuesday,False,Premium


**premium online orders: Nairobi and Mombasa**


1.   **Number of orders:** 83
2.   **Total revenue:** KES 10,218,000, about 14.2% of Duka's half-year revenue of KES 71.8M.
3.   **Export:** saved as `premium_online_coast_capital.csv` (without the index column).




In [9]:
import os
import pandas as pd

# Using the absolute path provided in the root directory
path = "/duka_branch_summary.xlsx"
if os.path.exists(path):
    xl = pd.ExcelFile(path)
    print("Sheet names:", xl.sheet_names)
else:
    print(f"Error: '{path}' was not found.")
    # Also checking if it is placed in /content/ or current working directory
    alt_path = "/content/duka_branch_summary.xlsx"
    if os.path.exists(alt_path):
        xl = pd.ExcelFile(alt_path)
        print(f"Found in alternate path. Sheet names:", xl.sheet_names)
    else:
        print("Current directory contents:", os.listdir('.'))

Error: '/duka_branch_summary.xlsx' was not found.
Current directory contents: ['.config', 'premium_online_coast_capital.csv', 'duka_sales.csv', 'drive', 'sample_data']


In [13]:
import os
import shutil

# Create the excel/ directory if it doesn't exist
os.makedirs("excel", exist_ok=True)

# Copy the file to the new destination
shutil.copy("/content/duka_branch_summary.xlsx", "excel/duka_branch_summary.xlsx")
print("File successfully copied to excel/duka_branch_summary.xlsx")

File successfully copied to excel/duka_branch_summary.xlsx


In [14]:
import pandas as pd

# Verify loading from the new path
summary = pd.read_excel("excel/duka_branch_summary.xlsx", sheet_name="Sheet1")

xl_totals = summary.dropna(subset=["branch"]).set_index("branch")["total-revenue(KES)"]
pd_totals = df.groupby("branch")["revenue"].sum()

xl_totals_clean = xl_totals.drop("Total", errors="ignore")

match = (xl_totals_clean.sort_index() == pd_totals.sort_index()).all()
print("Branch totals match pandas:", match)
print(f"Grand total Excel (without summary row): {xl_totals_clean.sum():,.0f} | pandas: {df['revenue'].sum():,.0f}")

Branch totals match pandas: False
Grand total Excel (without summary row): 71,792,800 | pandas: 71,792,500


In [12]:
import pandas as pd

# Corrected path to point to the file in /content/ where it is actually located
summary = pd.read_excel("/content/duka_branch_summary.xlsx", sheet_name="Sheet1")

xl_totals = summary.dropna(subset=["branch"]).set_index("branch")["total-revenue(KES)"]
pd_totals = df.groupby("branch")["revenue"].sum()

xl_totals_clean = xl_totals.drop("Total", errors="ignore")

match = (xl_totals_clean.sort_index() == pd_totals.sort_index()).all()
print("Branch totals match pandas:", match)
print(f"Grand total  Excel (without summary row): {xl_totals_clean.sum():,.0f} | pandas: {df['revenue'].sum():,.0f}")

Branch totals match pandas: False
Grand total  Excel (without summary row): 71,792,800 | pandas: 71,792,500


**spreadsheet reconciliation**

The branch revenue ranking was built in Google Sheets with `SUMIFS`, a region column with `XLOOKUP`, and a PivotTable of revenue by branch and channel, and it is saved as `excel/duka_branch_summary.xlsx`.

**Check:** the branch totals from the spreadsheet match pandas for all five branches, and both give a grand total of KES 71,792,500.

**executive summary**
- Duka Electronics earned **KES 71.8M** from 1,000 orders in January–June 2026, and **Nairobi contributed KES 26.1M (36.4%)** from 376 orders, more than three times Eldoret's 117.
- **Branch rankings are identical by revenue and by orders** (Nairobi, Mombasa, Nakuru, Kisumu, Eldoret) because average order value sits in a narrow KES 68k–76k band, so volume alone drives the ranking.
- **Laptops generated KES 41.5M (57.9% of revenue) from just 639 units**, while Tablets sold more units (691) but earned only KES 19.3M, so keeping laptops in stock matters more than chasing unit counts.
- **Weekends are slightly quieter per day, not weaker per order**: 277 weekend orders (5.3 per day) versus 723 weekday orders (5.6 per day), with near-identical average order values of KES 72.0k and KES 71.7k.
- **Revenue peaked in April at KES 13.5M and fell to KES 10.9M in June, a 24.0% gap**, while the 83 Premium online orders from Nairobi and Mombasa added KES 10.2M (14.2% of revenue).